In [ ]:
from pathlib import Path
import os
import re
import gc
import copy
import time
import json
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================================
# 0. 경로/설정 (여기만 먼저 수정)
# ============================================================
XLS_PATH = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(1)/candidate_metrics_bed00_bed10.csv")
IMG_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_filter(1)")
OUT_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/lettuce_heading_cnn_runs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# CSV 컬럼명
FILENAME_COL = "file_name"      # CSV 파일명 컬럼명. 필요시 "filename" 등으로 변경
BINARY_LABEL_COL = "core_lab_2class"   # 0/1 이진 라벨
MULTI_LABEL_COL = "core_strength"      # 0/1/2 다중 라벨

# 이미지
IMG_SIZE = 256
FILE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

# 학습
SEED = 42
N_SPLITS = 5
EPOCHS = 20
BATCH_SIZE = 16
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5
MIXED_PRECISION = True
MAX_GRAD_NORM = 1.0

# backbone: 성능/안정성 고려
BACKBONE = "efficientnet_b0"   # 추천 유지. 필요시 mobilenet_v3_small, resnet18 로 변경 가능
USE_PRETRAINED = True

# 재시작/중간저장
SAVE_EVERY_EPOCH = True
RESUME_IF_EXISTS = True

# 클래스 불균형 보정
USE_CLASS_WEIGHT = True

In [ ]:

# ============================================================
# 1. 공통 유틸
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def format_seconds(sec):
    sec = int(max(0, sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def write_log(log_path: Path, msg: str):
    print(msg)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def find_all_images(root: Path):
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in FILE_EXTS]


def build_image_index(root: Path):
    img_paths = find_all_images(root)
    idx = {}
    dup = {}
    for p in img_paths:
        name = p.name
        if name in idx:
            dup.setdefault(name, [idx[name]])
            dup[name].append(p)
        else:
            idx[name] = p
    return idx, dup, img_paths


def infer_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def get_model(backbone: str, num_classes: int, pretrained=True):
    backbone = backbone.lower()

    if backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model

    if backbone == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
        return model

    if backbone == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"지원하지 않는 BACKBONE: {backbone}")


def get_transforms(img_size=256):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=12),
        transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    valid_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, valid_tf


In [ ]:
# ============================================================
# 2. 데이터셋
# ============================================================
class LettuceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row["img_path"])
        label = int(row["label"])

        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, label, img_path.name


# ============================================================
# 3. 학습/검증 함수
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer, device, scaler=None, train=True):
    if train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            if scaler is not None and train:
                with torch.cuda.amp.autocast():
                    logits = model(images)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                if MAX_GRAD_NORM is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = criterion(logits, labels)
                if train:
                    loss.backward()
                    if MAX_GRAD_NORM is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    optimizer.step()

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_probs.extend(probs.detach().cpu().numpy().tolist())

    epoch_loss = running_loss / max(1, len(loader.dataset))
    return epoch_loss, np.array(all_labels), np.array(all_preds), np.array(all_probs)


def calc_metrics(y_true, y_pred, y_prob, num_classes, task_name):
    metrics = {}
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["f1_macro"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    metrics["f1_weighted"] = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

    if num_classes == 2:
        # positive class = 1
        try:
            metrics["auc"] = float(roc_auc_score(y_true, y_prob[:, 1]))
        except Exception:
            metrics["auc"] = np.nan
    else:
        try:
            metrics["auc_ovr_macro"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
        except Exception:
            metrics["auc_ovr_macro"] = np.nan

    metrics["confusion_matrix"] = confusion_matrix(y_true, y_pred).tolist()
    metrics["classification_report"] = classification_report(y_true, y_pred, zero_division=0)
    return metrics


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [ ]:
# ============================================================
# 4. 데이터 준비
# ============================================================
def prepare_dataframe(csv_path, img_root, filename_col, label_col, task_name, log_path):
    df = pd.read_csv(csv_path)

    if filename_col not in df.columns:
        raise ValueError(f"CSV에 {filename_col} 컬럼이 없습니다. 현재 컬럼: {list(df.columns)}")
    if label_col not in df.columns:
        raise ValueError(f"CSV에 {label_col} 컬럼이 없습니다. 현재 컬럼: {list(df.columns)}")

    img_index, dup, img_list = build_image_index(img_root)
    write_log(log_path, f"[이미지 인덱싱] 전체 이미지 수: {len(img_list):,}")
    write_log(log_path, f"[이미지 인덱싱] 중복 파일명 수: {len(dup):,}")

    work = df[[filename_col, label_col]].copy()
    work = work.rename(columns={filename_col: "file_name", label_col: "label"})
    work = work.dropna(subset=["file_name", "label"]).copy()
    work["file_name"] = work["file_name"].astype(str)
    work["label"] = work["label"].astype(int)

    keep_rows = []
    missing = []

    for _, row in work.iterrows():
        fname = row["file_name"]
        if fname in img_index:
            keep_rows.append({
                "file_name": fname,
                "label": int(row["label"]),
                "img_path": str(img_index[fname]),
            })
        else:
            missing.append(fname)

    out_df = pd.DataFrame(keep_rows)

    write_log(log_path, f"[{task_name}] CSV 행 수(결측 제거 후): {len(work):,}")
    write_log(log_path, f"[{task_name}] 매칭 성공 수: {len(out_df):,}")
    write_log(log_path, f"[{task_name}] 매칭 실패 수: {len(missing):,}")

    if len(missing) > 0:
        miss_path = OUT_ROOT / f"missing_{task_name}.txt"
        with open(miss_path, "w", encoding="utf-8") as f:
            for x in missing:
                f.write(str(x) + "\n")
        write_log(log_path, f"[{task_name}] 매칭 실패 파일명 저장: {miss_path}")

    if len(out_df) == 0:
        raise ValueError(f"[{task_name}] 매칭된 데이터가 없습니다.")

    cls_count = out_df["label"].value_counts().sort_index().to_dict()
    write_log(log_path, f"[{task_name}] 클래스 분포: {cls_count}")

    return out_df


# ============================================================
# 5. fold 학습
# ============================================================
def train_cv_task(df_all, task_name, num_classes, out_dir, device, log_path):
    out_dir.mkdir(parents=True, exist_ok=True)
    train_tf, valid_tf = get_transforms(IMG_SIZE)

    X = df_all["img_path"].values
    y = df_all["label"].values

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_results = []
    total_cv_start = time.time()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        fold_start = time.time()
        fold_dir = out_dir / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        ckpt_path = fold_dir / "best_model.pt"
        state_path = fold_dir / "resume_state.pt"
        result_path = fold_dir / "fold_result.json"

        train_df = df_all.iloc[tr_idx].reset_index(drop=True)
        valid_df = df_all.iloc[va_idx].reset_index(drop=True)

        write_log(log_path, "-" * 80)
        write_log(log_path, f"[{task_name}] Fold {fold}/{N_SPLITS} 시작 | train={len(train_df)} val={len(valid_df)}")
        write_log(log_path, f"[{task_name}] Fold {fold} train 분포: {train_df['label'].value_counts().sort_index().to_dict()}")
        write_log(log_path, f"[{task_name}] Fold {fold} valid 분포: {valid_df['label'].value_counts().sort_index().to_dict()}")

        train_ds = LettuceDataset(train_df, transform=train_tf)
        valid_ds = LettuceDataset(valid_df, transform=valid_tf)

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=False,
        )
        valid_loader = DataLoader(
            valid_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=False,
        )

        model = get_model(BACKBONE, num_classes=num_classes, pretrained=USE_PRETRAINED).to(device)

        # class weight
        class_weights = None
        if USE_CLASS_WEIGHT:
            counts = train_df["label"].value_counts().sort_index().to_dict()
            weight_list = []
            total = len(train_df)
            for c in range(num_classes):
                cnt = counts.get(c, 1)
                weight_list.append(total / (num_classes * cnt))
            class_weights = torch.tensor(weight_list, dtype=torch.float32, device=device)
            write_log(log_path, f"[{task_name}] Fold {fold} class_weights: {weight_list}")

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
        scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and MIXED_PRECISION))

        start_epoch = 1
        best_score = -np.inf
        best_payload = None
        no_improve = 0

        if RESUME_IF_EXISTS and state_path.exists():
            state = torch.load(state_path, map_location=device)
            model.load_state_dict(state["model_state"])
            optimizer.load_state_dict(state["optimizer_state"])
            scheduler.load_state_dict(state["scheduler_state"])
            if state.get("scaler_state") is not None and scaler is not None:
                scaler.load_state_dict(state["scaler_state"])
            start_epoch = state["epoch"] + 1
            best_score = state["best_score"]
            no_improve = state["no_improve"]
            write_log(log_path, f"[{task_name}] Fold {fold} resume from epoch {start_epoch}")

        for epoch in range(start_epoch, EPOCHS + 1):
            ep_start = time.time()

            tr_loss, tr_y, tr_pred, tr_prob = run_one_epoch(
                model, train_loader, criterion, optimizer, device, scaler=scaler, train=True
            )
            va_loss, va_y, va_pred, va_prob = run_one_epoch(
                model, valid_loader, criterion, optimizer, device, scaler=None, train=False
            )

            train_metrics = calc_metrics(tr_y, tr_pred, tr_prob, num_classes, task_name)
            valid_metrics = calc_metrics(va_y, va_pred, va_prob, num_classes, task_name)

            # 선택 기준
            valid_score = valid_metrics.get("auc", valid_metrics.get("auc_ovr_macro", valid_metrics["f1_macro"]))
            if np.isnan(valid_score):
                valid_score = valid_metrics["f1_macro"]

            scheduler.step(valid_score)

            ep_elapsed = time.time() - ep_start
            write_log(
                log_path,
                f"[{task_name}] Fold {fold} Epoch {epoch:02d}/{EPOCHS} | "
                f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} | "
                f"train_acc={train_metrics['accuracy']:.4f} val_acc={valid_metrics['accuracy']:.4f} | "
                f"train_f1m={train_metrics['f1_macro']:.4f} val_f1m={valid_metrics['f1_macro']:.4f} | "
                f"val_score={valid_score:.4f} | time={format_seconds(ep_elapsed)}"
            )

            if valid_score > best_score:
                best_score = valid_score
                no_improve = 0
                best_payload = {
                    "epoch": epoch,
                    "best_score": float(best_score),
                    "train_loss": float(tr_loss),
                    "valid_loss": float(va_loss),
                    "train_metrics": train_metrics,
                    "valid_metrics": valid_metrics,
                }
                torch.save({
                    "model_state": model.state_dict(),
                    "best_payload": best_payload,
                }, ckpt_path)
                write_log(log_path, f"[{task_name}] Fold {fold} best 갱신 -> {ckpt_path}")
            else:
                no_improve += 1

            if SAVE_EVERY_EPOCH:
                torch.save({
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "scaler_state": scaler.state_dict() if scaler is not None else None,
                    "best_score": best_score,
                    "no_improve": no_improve,
                }, state_path)

            if no_improve >= PATIENCE:
                write_log(log_path, f"[{task_name}] Fold {fold} early stopping (patience={PATIENCE})")
                break

        # best model 로드 후 fold 결과 저장
        saved = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(saved["model_state"])
        best_payload = saved["best_payload"]
        save_json(best_payload, result_path)

        fold_results.append({
            "fold": fold,
            **best_payload,
            "elapsed_sec": time.time() - fold_start,
        })

        write_log(log_path, f"[{task_name}] Fold {fold} 완료 | best_score={best_payload['best_score']:.4f} | fold_time={format_seconds(time.time()-fold_start)}")

        del model, train_loader, valid_loader, train_ds, valid_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # CV 요약
    summary_rows = []
    for x in fold_results:
        row = {
            "fold": x["fold"],
            "epoch": x["epoch"],
            "best_score": x["best_score"],
            "train_loss": x["train_loss"],
            "valid_loss": x["valid_loss"],
            "train_acc": x["train_metrics"]["accuracy"],
            "valid_acc": x["valid_metrics"]["accuracy"],
            "train_f1_macro": x["train_metrics"]["f1_macro"],
            "valid_f1_macro": x["valid_metrics"]["f1_macro"],
            "elapsed_sec": x["elapsed_sec"],
        }
        if num_classes == 2:
            row["valid_auc"] = x["valid_metrics"].get("auc", np.nan)
        else:
            row["valid_auc_ovr_macro"] = x["valid_metrics"].get("auc_ovr_macro", np.nan)
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = out_dir / f"cv_summary_{task_name}.csv"
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

    mean_dict = summary_df.mean(numeric_only=True).to_dict()
    save_json(mean_dict, out_dir / f"cv_mean_{task_name}.json")

    write_log(log_path, "=" * 80)
    write_log(log_path, f"[{task_name}] CV 완료 | 총 시간={format_seconds(time.time() - total_cv_start)}")
    write_log(log_path, f"[{task_name}] 요약 CSV 저장: {summary_csv}")
    write_log(log_path, f"[{task_name}] 평균 성능: {mean_dict}")

    return summary_df



In [ ]:

# ============================================================
# 6. 메인
# ============================================================
def main():
    seed_everything(SEED)
    device = infer_device()

    run_name = f"run_{time.strftime('%Y%m%d_%H%M%S')}_{BACKBONE}_img{IMG_SIZE}"
    run_dir = OUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / "train_log.txt"

    write_log(log_path, "=" * 100)
    write_log(log_path, f"[START] {now_str()}")
    write_log(log_path, f"CSV_PATH   : {CSV_PATH}")
    write_log(log_path, f"IMG_ROOT   : {IMG_ROOT}")
    write_log(log_path, f"OUT_ROOT   : {run_dir}")
    write_log(log_path, f"DEVICE     : {device}")
    write_log(log_path, f"BACKBONE   : {BACKBONE}")
    write_log(log_path, f"IMG_SIZE   : {IMG_SIZE}")
    write_log(log_path, f"BATCH_SIZE : {BATCH_SIZE}")
    write_log(log_path, f"EPOCHS     : {EPOCHS}")
    write_log(log_path, f"N_SPLITS   : {N_SPLITS}")
    write_log(log_path, f"LR         : {LR}")

    # ----------------------
    # 실험 1: binary
    # ----------------------
    binary_df = prepare_dataframe(
        csv_path=CSV_PATH,
        img_root=IMG_ROOT,
        filename_col=FILENAME_COL,
        label_col=BINARY_LABEL_COL,
        task_name="binary_heading",
        log_path=log_path,
    )
    binary_out_dir = run_dir / "binary_heading"
    binary_summary = train_cv_task(
        df_all=binary_df,
        task_name="binary_heading",
        num_classes=2,
        out_dir=binary_out_dir,
        device=device,
        log_path=log_path,
    )

    # ----------------------
    # 실험 2: multiclass
    # ----------------------
    multi_df = prepare_dataframe(
        csv_path=CSV_PATH,
        img_root=IMG_ROOT,
        filename_col=FILENAME_COL,
        label_col=MULTI_LABEL_COL,
        task_name="multiclass_core_strength",
        log_path=log_path,
    )
    multi_out_dir = run_dir / "multiclass_core_strength"
    multi_summary = train_cv_task(
        df_all=multi_df,
        task_name="multiclass_core_strength",
        num_classes=3,
        out_dir=multi_out_dir,
        device=device,
        log_path=log_path,
    )

    # 최종 비교 요약
    compare = {
        "binary_heading_mean": binary_summary.mean(numeric_only=True).to_dict(),
        "multiclass_core_strength_mean": multi_summary.mean(numeric_only=True).to_dict(),
    }
    save_json(compare, run_dir / "final_compare_summary.json")

    write_log(log_path, "=" * 100)
    write_log(log_path, f"[END] {now_str()}")
    write_log(log_path, f"최종 비교 저장: {run_dir / 'final_compare_summary.json'}")
    write_log(log_path, f"전체 결과 폴더: {run_dir}")
    write_log(log_path, "완료.")



In [ ]:

if __name__ == "__main__":
    main()


[START] 2026-03-15 15:14:55
CSV_PATH   : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_coretest(1)/candidate_metrics_bed00_bed10.csv
IMG_ROOT   : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/crops_filter(1)
OUT_ROOT   : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/lettuce_heading_cnn_runs/run_20260315_151455_efficientnet_b0_img256
DEVICE     : cuda
BACKBONE   : efficientnet_b0
IMG_SIZE   : 256
BATCH_SIZE : 16
EPOCHS     : 20
N_SPLITS   : 5
LR         : 0.0001
[이미지 인덱싱] 전체 이미지 수: 1,088
[이미지 인덱싱] 중복 파일명 수: 0
[binary_heading] CSV 행 수(결측 제거 후): 273
[binary_heading] 매칭 성공 수: 273
[binary_heading] 매칭 실패 수: 0
[binary_heading] 클래스 분포: {0: 94, 1: 179}
------------------------------------------------------

100%|██████████| 20.5M/20.5M [00:00<00:00, 148MB/s]


[binary_heading] Fold 1 class_weights: [1.4533333333333334, 0.7622377622377622]


/tmp/ipykernel_1448/3703839085.py:126: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and MIXED_PRECISION))
/tmp/ipykernel_1448/4130289603.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[binary_heading] Fold 1 Epoch 01/20 | train_loss=0.6872 val_loss=0.6967 | train_acc=0.5092 val_acc=0.4545 | train_f1m=0.5062 val_f1m=0.4544 | val_score=0.5117 | time=00:00:52
[binary_heading] Fold 1 best 갱신 -> /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/lettuce_heading_cnn_runs/run_20260315_151455_efficientnet_b0_img256/binary_heading/fold_1/best_model.pt


...

[multiclass_core_strength] Fold 5 Epoch 13/20 | train_loss=0.2654 val_loss=1.2648 | train_acc=0.8767 val_acc=0.5556 | train_f1m=0.8864 val_f1m=0.4606 | val_score=0.7083 | time=00:00:02


/tmp/ipykernel_1448/4130289603.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[multiclass_core_strength] Fold 5 Epoch 14/20 | train_loss=0.1907 val_loss=1.3263 | train_acc=0.9589 val_acc=0.5370 | train_f1m=0.9552 val_f1m=0.4376 | val_score=0.7021 | time=00:00:02


/tmp/ipykernel_1448/4130289603.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[multiclass_core_strength] Fold 5 Epoch 15/20 | train_loss=0.1973 val_loss=1.3666 | train_acc=0.9087 val_acc=0.5185 | train_f1m=0.9036 val_f1m=0.4256 | val_score=0.6930 | time=00:00:02
[multiclass_core_strength] Fold 5 early stopping (patience=5)
[multiclass_core_strength] Fold 5 완료 | best_score=0.7718 | fold_time=00:00:52
[multiclass_core_strength] CV 완료 | 총 시간=00:04:59
[multiclass_core_strength] 요약 CSV 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/lettuce_heading_cnn_runs/run_20260315_151455_efficientnet_b0_img256/multiclass_core_strength/cv_summary_multiclass_core_strength.csv
[multiclass_core_strength] 평균 성능: {'fold': 3.0, 'epoch': 13.0, 'best_score': 0.7945188529000984, 'train_loss': 0.3383400649053613, 'valid_loss': 0.9805353366806852, 'train_acc': 0.8811570524904695, 'valid_acc': 0.6517171717171718, 'train_f1_macro': 0.8825136481379345, 'valid_f1_macro': 0.5667238684566411, 'elaps